In [1]:
from budget import load_budget, extract_cashflow, aggregate_cashflow
from cpiu import load_cpiu

from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
from plotly.subplots import make_subplots
from scipy.stats import norm
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import polars.selectors as cs

In [2]:
cpiu = load_cpiu()
budget = load_budget()

In [3]:
import datetime

if datetime.date.today() >= datetime.date(2026, 2, 1):
    raise Exception("Elevator expenditure priced in")
ELEVATOR_EXPENDITURE = 31247.13
ELEVATOR_EXPENDITURE_EXPR: pl.Expr = (
    pl.when(pl.col("month").is_between(1, 3))
    .then(ELEVATOR_EXPENDITURE / 3)
    .otherwise(0)
)

In [4]:
expenses = extract_cashflow(budget["expenses"])


# Simulate next year.
MONTHS_TO_SIMULATE = 12
SIMULATIONS = 10000
UNPAID_BILLS = 3146.42
# Based on the "Operation" account pulled from Daisy dashboard on 2026-01-10
STARTING_BALANCE = 23080.09 - UNPAID_BILLS
cashflow_by_month = (
    aggregate_cashflow(extract_cashflow(budget["incomes"]))
    .with_columns(
        pl.int_ranges(-3, 9)
        .alias("budget_increase")
        .list.eval((pl.element() / 100).round(2))
    )
    .explode("budget_increase")
    .with_columns(pl.col("amount") * (1 + pl.col("budget_increase")))
    .join(
        aggregate_cashflow(expenses)
        .join(cpiu, left_on=["year", "month"], right_on=["Year", "month"])
        .select("year", "month", pl.col("amount") * (1 + pl.col("eoy_delta"))),
        on=["year", "month"],
        how="full",
        suffix="_expense",
    )
    .select(
        "budget_increase",
        "year",
        "month",
        pl.col("amount").fill_null(0) - pl.col("amount_expense").fill_null(0),
    )
)
monte_carlo_simulations = (
    cashflow_by_month.group_by("budget_increase")
    .agg(pl.col("amount"))
    .with_columns(simulation_id=pl.int_ranges(0, SIMULATIONS))
    .explode("simulation_id")
    .with_columns(pl.col("amount").list.sample(MONTHS_TO_SIMULATE, shuffle=True))
    .explode("amount")
    .with_columns(
        (pl.row_index("month").over("budget_increase", "simulation_id") + 1).cast(
            pl.UInt8
        )
    )
    .with_columns(
        (pl.col("amount") - ELEVATOR_EXPENDITURE_EXPR)
        .cum_sum()
        .over("budget_increase", "simulation_id")
        + STARTING_BALANCE,
    )
)

In [5]:
def line_charts(simulations: pl.DataFrame) -> go.Figure:
    simulations_by_month = (
        simulations.group_by("month", "budget_increase")
        .agg(
            amount_average=pl.col("amount").mean(),
            amount_min=pl.col("amount").min(),
            amount_max=pl.col("amount").max(),
        )
        .sort("budget_increase", "month")
    )

    TRENDLINE_FIG_COL_COUNT = 2

    unique_budget_increases = simulations["budget_increase"].unique()

    fig = make_subplots(
        rows=int(len(unique_budget_increases) / TRENDLINE_FIG_COL_COUNT),
        cols=TRENDLINE_FIG_COL_COUNT,
        shared_yaxes="all",
        subplot_titles=[
            f"budget_increase={budget_increase}"
            for budget_increase in unique_budget_increases
        ],
    )
    for i, budget_increase in (
        unique_budget_increases.to_frame().with_row_index().iter_rows()
    ):
        sims = simulations_by_month.filter(pl.col("budget_increase") == budget_increase)
        row = int(i / TRENDLINE_FIG_COL_COUNT) + 1
        col = i % TRENDLINE_FIG_COL_COUNT + 1
        fig.add_trace(
            go.Scatter(
                x=pl.concat([sims["month"], sims["month"].reverse()]),
                y=pl.concat([sims["amount_min"], sims["amount_max"].reverse()]),
                name=f"min/max {budget_increase}",
                fill="toself",
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(x=sims["month"], y=sims["amount_average"], name=budget_increase),
            row=row,
            col=col,
        )
        fig.update_xaxes(title_text="month", row=row, col=col)
    fig.update_layout(height=1000, legend=go.layout.Legend(title="budget_increase"))
    return fig


In [6]:
line_charts(monte_carlo_simulations).update_layout(
    title_text="Monte Carlo Simulations"
).show()

In [7]:
def pie_charts(simulations: pl.DataFrame) -> go.Figure:
    simulation_mins = simulations.group_by("budget_increase", "simulation_id").agg(
        pl.col("amount").min()
    )
    fig = px.pie(
        simulation_mins.group_by(
            "budget_increase",
            ruinous=pl.col("amount") < 0,
        )
        .len("simulation_count")
        .with_columns(
            ruinous=pl.when("ruinous")
            .then(pl.lit("Special Assessment"))
            .otherwise(pl.lit("Safe"))
        )
        .sort("budget_increase"),
        names="ruinous",
        values="simulation_count",
        facet_col="budget_increase",
        facet_col_wrap=3,
        color_discrete_sequence=["#4B08AF", "#32965D"],
        height=1000,
    )
    return fig


pie_charts(monte_carlo_simulations).update_layout(
    title="Likelihood of Special Assessment (Monte Carlo)",
).show(renderer="notebook_connected")

# Special Assessment
95% confidence that the special assessment--if there is one--will be less than `amount`

In [8]:
# Inflation estimation from https://www.federalreserve.gov/monetarypolicy/files/fomcprojtabl20250917.pdf
INFLATION = 0.026
CONFIDENCE_LEVELS = [0.95, 0.99]
assessment_recommendations = (
    monte_carlo_simulations.group_by("budget_increase", "simulation_id")
    .agg(pl.col("amount").min())
    .group_by("budget_increase")
    .agg(
        -pl.col("amount").quantile(1 - confidence_level).alias(str(confidence_level))
        for confidence_level in CONFIDENCE_LEVELS
    )
    .with_columns(
        (
            pl.col(str(c) for c in CONFIDENCE_LEVELS).name.suffix("_with_inflation")
            * (1 + INFLATION)
        )
    )
    .unpivot(
        index="budget_increase", variable_name="confidence", value_name="assessment"
    )
)

In [9]:
px.bar(
    assessment_recommendations.sort("confidence"),
    "budget_increase",
    "assessment",
    "confidence",
    barmode="group",
)

In [10]:
budget_surface = (
    monte_carlo_simulations.group_by("budget_increase", "simulation_id")
    .agg(pl.col("amount").min())
    .with_columns(-pl.col("amount"))
    .sort("budget_increase", "amount")
    .with_columns(
        percentile=(
            pl.col("amount").rank().over("budget_increase")
            / (pl.len().over("budget_increase") + 1)
        )
    )
    .with_columns((pl.col("amount") / 1000).cast(pl.Int64))
    .with_columns(pl.col("amount") * 1000)
    .pivot(
        index="amount",
        on="budget_increase",
        values="percentile",
        aggregate_function="mean",
    )
    .sort("amount")
    .with_columns(cs.exclude(cs.first()).forward_fill().backward_fill())
)
go.Figure(
    data=[
        go.Surface(
            x=budget_surface.columns[1:],
            y=budget_surface["amount"],
            z=budget_surface.select(budget_surface.columns[1:]),
        ),
        go.Surface(
            x=budget_surface.columns[1:],
            y=budget_surface["amount"] * (1 + INFLATION),
            z=budget_surface.select(budget_surface.columns[1:]),
        ),
    ]
).update_layout(
    scene=go.layout.Scene(
        xaxis=go.layout.scene.XAxis(title="budget increase"),
        yaxis=go.layout.scene.YAxis(title="special assessment"),
        zaxis=go.layout.scene.ZAxis(title="probability of success"),
    )
)

In [11]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="headline_category",
)
fig.show(renderer="notebook_connected")

In [12]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="category",
)
fig.show(renderer="notebook_connected")